In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from deeppic import *

In [2]:
def standardize(X: torch.Tensor) -> torch.Tensor:
    return (X - X.mean(dim=0, keepdim=True)) / X.std(dim=0, unbiased=False, keepdim=True)

def linear_data(n: int, p: int, s: int):
    X = standardize(torch.randn(n, p, dtype=torch.float32))
    beta = torch.zeros(p, dtype=torch.float32)
    true_features = torch.randperm(p)[:s]
    beta[true_features] = 3 * torch.ones(s, dtype=torch.float32)
    y = X @ beta + torch.randn(n, dtype=torch.float32)
    return X, y, true_features.tolist()

def nonlinear_data(n: int, p: int, s: int, snr: float = 5.0):
    X = standardize(torch.randn(n, p, dtype=torch.float32))
    true_features = torch.randperm(p)[:s]
    X_s = X[:, true_features]
    y = 0
    for i in range(s):
        y += snr * np.abs(X_s[:, i])
    y = y + torch.randn(n, dtype=torch.float32)
    return X, y, true_features.tolist()

def hard_nonlinear_data(n: int, p: int, snr: float = 5.0):
    X = standardize(torch.randn(n, p, dtype=torch.float32))
    true_features = torch.randperm(p)[:4]
    X_s = X[:, true_features]
    mu = snr * np.abs(
        np.abs(X_s[:, 1] - X_s[:, 0])
      - np.abs(X_s[:, 3] - X_s[:, 2])
    )
    y = mu + torch.randn(n, dtype=torch.float32)
    return X, y, true_features.tolist()


_SHAPES = {
    # name        f(x)                                    odd?  strongly nonlinear?
    "cubic":      lambda x: x**3,
    "exp":        lambda x: torch.exp(x) - np.e**0.5,
    "tanh":       lambda x: torch.tanh(3*x),
    "sqrt_abs":   lambda x: torch.sign(x)*x.abs().sqrt(),
    "sigmoid":    lambda x: 1/(1+torch.exp(-4*x)),
    "piecewise":  lambda x: torch.where(x > 0, 2*x, 0.2*x),      # kinked, monotone
}

def monotone_nonlinear_data(n, p, s, shape="cubic", snr=5.0, seed=None):
    """Additive nonlinear signal with NON-ZERO average slope.

    y = c * sum_{j in S} f(X_j) + eps,  with f odd or monotone, so that
    E[dy/dX_j] != 0 and the feature is visible from the null model.
    `c` is set so that sd(signal)/sd(noise) = snr.
    """
    if seed is not None:
        torch.manual_seed(seed)
    f = _SHAPES[shape]
    X = standardize(torch.randn(n, p))
    if s != 0:
        true = torch.randperm(p)[:s]
        mu = f(X[:, true]).sum(1)
        mu = (mu - mu.mean()) / mu.std(unbiased=False)
    else:
        true = torch.tensor([])
        mu = 0
    eps = torch.randn(n)
    return X, snr*mu + eps, sorted(true.tolist())


In [9]:
n, p, s = 700, 100, 4
X, y, true = hard_nonlinear_data(n, p, snr=10.0)

model = SparseRegressor(hidden_dims=(32, 16, 8))
model.fit(X, y, verbose=True)
model.selected_

Calibrating lambda at level alpha = 0.05 (1000 replicates)...
lambda      0.129038  (calibrated)
lambda_data 0.148610  (the null is stationary above this)
Path        5 phases, warm-started, from lam = 0.025808
Anneal      one cosine per phase, starting from 1.00e-02 and shrinking by 0.75
            each phase, down to 0.001 of that within the phase
Stop        at least 100 iterations, and the objective moving by
            less than 0.0001 in relative terms over 50 iterations

Phase 1/5  lam = 0.025808  lr = 1.00e-02
    iter        loss     penalty    objective       dF/F    gauge   sel
    ----  ----------  ----------  -----------  ---------  -------  ----
       1     11.5725      0.4323      12.0048        inf   0.8706    62
       2     11.5159      0.0109      11.5268        inf   1.0634    44
       3     11.4490      0.0130      11.4620        inf   1.2360    39
       4     11.3666      0.0204      11.3870        inf   1.1133    37
       5     11.2798      0.0253      11.3

[67]

In [3]:
n, p, s = 1700, 100, 4
X = torch.randn(size=(n, p), dtype=torch.float32)
mu = torch.relu(
        torch.relu(X[:, 0] + X[:, 1]) * torch.relu(X[:, 2] - X[:, 3]) - 1.0
    )
y = mu + + torch.randn(size=(n,))

model = SparseRegressor(hidden_dims=(32, 16))
model.fit(X, y, verbose=True)
model.selected_

Calibrating lambda at level alpha = 0.05 (1000 replicates)...
lambda      0.084869  (calibrated)
lambda_data 0.119391  (the null is stationary above this)
Path        5 phases, warm-started, from lam = 0.016974
Anneal      one cosine per phase, starting from 1.00e-02 and shrinking by 0.75
            each phase, down to 0.001 of that within the phase
Stop        at least 100 iterations, and the objective moving by
            less than 0.0001 in relative terms over 50 iterations

Phase 1/5  lam = 0.016974  lr = 1.00e-02
    iter        loss     penalty    objective       dF/F    gauge   sel
    ----  ----------  ----------  -----------  ---------  -------  ----
       1      1.1671      0.3046       1.4717        inf   0.9047    80
       2      1.1444      0.0163       1.1607        inf   1.0269    61
       3      1.1357      0.0179       1.1536        inf   1.0471    56
       4      1.1275      0.0213       1.1488        inf   1.0615    50
       5      1.1195      0.0237       1.1

[0, 1, 2, 3]

In [4]:
model.model_

VariableSelectionMLP(
  (network): Sequential(
    (0): Linear(in_features=100, out_features=32, bias=True)
    (1): LeakyReLU(negative_slope=0.1)
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): LeakyELU(m=0.1)
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)

In [ ]:
def linear_binary_class_data(n: int, p: int, s: int):
    X = standardize(torch.randn(n, p, dtype=torch.float32))
    beta = torch.zeros(p, dtype=torch.float32)
    true_features = torch.randperm(p)[:s]
    beta[true_features] = 3 * torch.ones(s, dtype=torch.float32)
    proba = torch.sigmoid(X @ beta)
    y = torch.bernoulli(proba)
    return X, y, true_features.tolist()

n, p, s = 300, 100, 0
l = []
for _ in range(50):
    X, y, true = linear_binary_class_data(n, p, s)

    model = SparseClassifier(hidden_dims=None)
    model.fit(X, y)
    l.append(set(model.selected_)==set(true))

np.mean(l)

In [58]:
def components(x: torch.Tensor) -> torch.Tensor:
    """The four additive components, evaluated columnwise on ``(n, 4)``."""
    return torch.stack(
        [
            -2.0 * torch.sin(2.0 * x[:, 0]),
            x[:, 1] ** 2 - 1.0 / 3.0,
            x[:, 2] - 0.5,
            torch.exp(-x[:, 3]) + np.exp(-1.0) - 1.0,
        ],
        dim=1,
    )


def make_dataset(n, p):
    X = torch.randn(n, p)
    true = torch.randperm(p)[:4]
    y = components(X[:, true]).sum(dim=1) + torch.randn(n)

    return X, y, sorted(true.tolist())

X, y, true = make_dataset(100, 128)

model = SparseRegressor(hidden_dims=(10, 5))
model.fit(X, y, verbose=True)
print(model.selected_, true)

Calibrating lambda at level alpha = 0.05 (1000 replicates)...
lambda      0.357818  (calibrated)
lambda_data 0.535803  (the null is stationary above this)
Path        5 phases, warm-started, from lam = 0.071564
Anneal      one cosine per phase, starting from 1.00e-02 and shrinking by 0.75
            each phase, down to 0.001 of that within the phase
Stop        at least 100 iterations, and the objective moving by
            less than 0.0001 in relative terms over 50 iterations

Phase 1/5  lam = 0.071564  lr = 1.00e-02
    iter        loss     penalty    objective       dF/F    gauge   sel
    ----  ----------  ----------  -----------  ---------  -------  ----
       1      3.3711      0.5811       3.9522        inf   0.8489    80
      21      3.1734      0.0333       3.2067        inf   1.0372    15
      41      2.5872      0.1598       2.7471        inf   1.0473    26
      61      1.9290      0.4862       2.4152   3.60e-01   1.0276    46
      81      1.4180      0.7620       2.1

In [135]:
def make_data(n, p, s, gamma=5.0, sigma=1.0, rng=None):
    rng = np.random.default_rng(rng)
    Z = rng.standard_normal((n, p))
    S = np.sort(rng.choice(p, size=s, replace=False))

    signal = gamma * np.abs(Z[:, S]).sum(axis=1)
    y = signal + sigma * rng.standard_normal(n)

    X = Z
    return X, y, S

def make_data2(n, p, s):
    X = torch.randn(n, p, dtype=torch.float32)
    beta = torch.zeros(p, dtype=torch.float32)
    true_features = torch.randperm(p)[:s]
    beta[true_features] = 3 * torch.ones(s, dtype=torch.float32)
    y = torch.tanh(X @ beta) + torch.randn(n, dtype=torch.float32)
    return X, y, true_features.tolist()


l = []
for _ in range(20):
    X, y, true = make_data(1200, 50, 12)
    model = SparseRegressor(hidden_dims=(32, ))
    model.fit(X, y)
    l.append(set(model.selected_)==set(true))

np.mean(l)

0.15